In [1]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pymongo

In [6]:
from pymongo import MongoClient
import json

## Import the hospital.json file into a collection named hospital

# 1. Connect to MongoDB (localhost by default)
client = MongoClient("mongodb://localhost:27017/")

# 2. Create/use a new database called eCommerceDB
eCommerce_db = client["eCommerceDB"]

# 3. Create/use a collection called hospital inside eCommerceDB
eCommerce_collection = hospital_db["eCommerce"]

# 4. Load hospital.json data
with open("ecomm.json") as f:
    data = json.load(f)

# 5. Insert data (supports both single or multiple documents)
eCommerce_collection.insert_many(data)

print("The e-Commerce data inserted successfully!")

The e-Commerce data inserted successfully!


In [8]:
## i) Insert one document of nested levels  [single_ecomm_doc.json]

# 1. Load single_ecomm_doc.json data
with open("single_ecomm_doc.json") as f:
    data = json.load(f)

# 2. Insert data (supports both single or multiple documents)
eCommerce_collection.insert_one(data)

print("The eCommerce data inserted successfully!")

The eCommerce data inserted successfully!


In [9]:
## ii) Insert multiple documents of nested levels  [multiple_ecomm_doc.json]

# 1. Load multiple_ecomm_doc.json data
with open("multiple_ecomm_doc.json") as f:
    data = json.load(f)

# 2. Insert data (supports both single or multiple documents)
eCommerce_collection.insert_many(data)

print("The eCommerce data inserted successfully!")

The eCommerce data inserted successfully!


In [15]:
## iii) Delete a customer who has viewed products of Electronics category, 
## has a wish list category of Books and has placed an order where the total price is greater than 1000.

eCommerce_collection.delete_one({
    "viewedProducts.category": "Electronics",
    "wishlist.category": "Books",
    "orderHistory.totalPrice": {"$gt": 1000}
})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

In [16]:
## iv) Delete customers living in MD and having order history of less than 1000.

eCommerce_collection.delete_many({
    "personalInfo.contact.address.state": "MD",
    "orderHistory.totalPrice": {"$lt": 1000}
})

DeleteResult({'n': 7, 'ok': 1.0}, acknowledged=True)

In [17]:
## v) If a customer has viewed products of category book with cart items having price greater than 100, 
## change his email and set first item in his wish list priority of High.

eCommerce_collection.update_many(
    {"viewedProducts.category": "Books",
     "cart.items.items.details.specs.price": {"$gt": 100} },
    {"$set":
        {"personalInfo.contact.email": "updated_to_new@example.com",
         "wishlist.0.priority": "High"}
    }
)

UpdateResult({'n': 8, 'nModified': 8, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [34]:
## vi) Find customers who have items in cart greater than $100 and have viewed Electronic Products.


results = eCommerce_collection.find({
    "cart.items.items.details.specs.price": {"$gt": 100},
    "viewedProducts.category": "Electronics"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c1106'), 'customerID': 'CUST0007', 'personalInfo': {'name': {'firstName': 'Prady_7', 'lastName': 'Mohta_7'}, 'contact': {'email': 'prady7@example.com', 'phone': '731-555-5317', 'address': {'street': '284 Guliford Road', 'city': 'Bethesda', 'state': 'VA', 'zipCode': '61151', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Home Appliances', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 112, 'features': {'dimension': '13 inches', 'weight': '5 kg', 'extras': {'warranty': '1 year', 'color': 'White'}}}}}]}, {'category': 'Clothing', 'subCategory': 'Subcategory 2', 'ite

In [45]:
# print pretty for question vi

import json
from bson import json_util

results_brief = eCommerce_collection.find(
    {"cart.items.items.details.specs.price": {"$gt": 100},
     "viewedProducts.category": "Electronics"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find(
    {"cart.items.items.details.specs.price": {"$gt": 100},
     "viewedProducts.category": "Electronics"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "cart.items.items.details.specs.price": 1,
     "viewedProducts.category": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0007"
}
{
  "customerID": "CUST0013"
}
{
  "customerID": "CUST0015"
}
{
  "customerID": "CUST0021"
}
{
  "customerID": "CUST0024"
}
{
  "customerID": "CUST0025"
}
{
  "customerID": "CUST0007",
  "viewedProducts": [
    {
      "category": "Home Appliances"
    },
    {
      "category": "Clothing"
    },
    {
      "category": "Footwear"
    },
    {
      "category": "Electronics"
    }
  ],
  "cart": {
    "items": [
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 551
              }
            }
          }
        ]
      },
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 653
              }
            }
          }
        ]
      },
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 161
              }
            }
          }
        ]
      }
    ]
  }
}
{
  "

In [36]:
## vii) Find customers who are from MD and have viewed products of category both Electronics and Books.

results = eCommerce_collection.find({
    "personalInfo.contact.address.state": "MD",
    "viewedProducts.category": {"$all":["Electronics","Books"]}
})

for result in results:
    print(result)

{
  "_id": {
    "$oid": "68f45a4f0ba48fd48e8c110e"
  },
  "customerID": "CUST0015",
  "personalInfo": {
    "name": {
      "firstName": "Prady_15",
      "lastName": "Mohta_15"
    },
    "contact": {
      "email": "updated_to_new@example.com",
      "phone": "287-555-4363",
      "address": {
        "street": "885 Guliford Road",
        "city": "Adelphi",
        "state": "MD",
        "zipCode": "20507",
        "country": "USA",
        "deliveryDetails": {
          "instructions": "Leave at front door",
          "timePreferences": {
            "weekdays": "Afternoon",
            "weekends": "Morning",
            "specialInstructions": {
              "ifNotHome": "Leave with neighbor",
              "neighborDetails": {
                "name": "Manav Gupta",
                "contactNumber": "333-444-5555",
                "address": "4321 Hartwick Road"
              }
            }
          }
        }
      }
    }
  },
  "viewedProducts": [
    {
      "category": "El

In [46]:
# print pretty for question vii

results_brief = eCommerce_collection.find({
    "personalInfo.contact.address.state": "MD",
    "viewedProducts.category": {"$all":["Electronics","Books"]}},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "personalInfo.contact.address.state": "MD",
    "viewedProducts.category": {"$all":["Electronics","Books"]}},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "personalInfo.contact.address.state": 1,
     "viewedProducts.category": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0015"
}
{
  "customerID": "CUST0015",
  "personalInfo": {
    "contact": {
      "address": {
        "state": "MD"
      }
    }
  },
  "viewedProducts": [
    {
      "category": "Electronics"
    },
    {
      "category": "Books"
    }
  ]
}


In [47]:
## viii) Find customers who have products of Footwear in their wish list and have viewed Home Appliances. 

results = eCommerce_collection.find({
    "wishlist.category": "Footwear",
    "viewedProducts.category": "Home Appliances"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c1100'), 'customerID': 'CUST0001', 'personalInfo': {'name': {'firstName': 'Prady_1', 'lastName': 'Mohta_1'}, 'contact': {'email': 'prady1@example.com', 'phone': '810-555-5903', 'address': {'street': '968 Guliford Road', 'city': 'Bethesda', 'state': 'VA', 'zipCode': '49836', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Clothing', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 372, 'features': {'dimension': '6 inches', 'weight': '3 kg', 'extras': {'warranty': '1 year', 'color': 'White'}}}}}]}, {'category': 'Footwear', 'subCategory': 'Subcategory 2', 'items': [{'

In [49]:
# print pretty for question viii

results_brief = eCommerce_collection.find({
    "wishlist.category": "Footwear",
    "viewedProducts.category": "Home Appliances"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "wishlist.category": "Footwear",
    "viewedProducts.category": "Home Appliances"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "wishlist.category": 1,
     "viewedProducts.category": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0001"
}
{
  "customerID": "CUST0005"
}
{
  "customerID": "CUST0008"
}
{
  "customerID": "CUST0017"
}
{
  "customerID": "CUST0022"
}
{
  "customerID": "CUST0001",
  "viewedProducts": [
    {
      "category": "Clothing"
    },
    {
      "category": "Footwear"
    },
    {
      "category": "Home Appliances"
    }
  ],
  "wishlist": [
    {
      "category": "Clothing"
    },
    {
      "category": "Footwear"
    }
  ]
}
{
  "customerID": "CUST0005",
  "viewedProducts": [
    {
      "category": "Footwear"
    },
    {
      "category": "Home Appliances"
    },
    {
      "category": "Home Appliances"
    }
  ],
  "wishlist": [
    {
      "category": "Footwear"
    }
  ]
}
{
  "customerID": "CUST0008",
  "viewedProducts": [
    {
      "category": "Clothing"
    },
    {
      "category": "Home Appliances"
    },
    {
      "category": "Books"
    },
    {
      "category": "Clothing"
    }
  ],
  "wishlist": [
    {
      "category": "Home Appliances"
    },

In [50]:
## ix)  Find Customers who have ordered more than $1500 goods and live in College Park.

results = eCommerce_collection.find({
    "orderHistory.totalPrice": {"$gt": 1500},
    "personalInfo.contact.address.city": "College Park"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c110f'), 'customerID': 'CUST0016', 'personalInfo': {'name': {'firstName': 'Prady_16', 'lastName': 'Mohta_16'}, 'contact': {'email': 'prady16@example.com', 'phone': '599-555-7765', 'address': {'street': '510 Guliford Road', 'city': 'College Park', 'state': 'VA', 'zipCode': '24412', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Home Appliances', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 129, 'features': {'dimension': '15 inches', 'weight': '4 kg', 'extras': {'warranty': '1 year', 'color': 'Silver'}}}}}]}, {'category': 'Clothing', 'subCategory': 'Subcategory 

In [51]:
# print pretty for question ix

results_brief = eCommerce_collection.find({
    "orderHistory.totalPrice": {"$gt": 1500},
    "personalInfo.contact.address.city": "College Park"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "orderHistory.totalPrice": {"$gt": 1500},
    "personalInfo.contact.address.city": "College Park"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "orderHistory.totalPrice": 1,
     "personalInfo.contact.address.city": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0016"
}
{
  "customerID": "CUST0020"
}
{
  "customerID": "CUST0016",
  "personalInfo": {
    "contact": {
      "address": {
        "city": "College Park"
      }
    }
  },
  "orderHistory": [
    {
      "totalPrice": 1679
    },
    {
      "totalPrice": 946
    }
  ]
}
{
  "customerID": "CUST0020",
  "personalInfo": {
    "contact": {
      "address": {
        "city": "College Park"
      }
    }
  },
  "orderHistory": [
    {
      "totalPrice": 1142
    },
    {
      "totalPrice": 1787
    }
  ]
}


In [52]:
## x) Find customers who have viewed Electronic Products and have items which are of Electronics category in their cart.

results = eCommerce_collection.find({
    "viewedProducts.category": "Electronics",
    "cart.items.category": "Electronics"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c1117'), 'customerID': 'CUST0024', 'personalInfo': {'name': {'firstName': 'Prady_24', 'lastName': 'Mohta_24'}, 'contact': {'email': 'prady24@example.com', 'phone': '829-555-4892', 'address': {'street': '812 Guliford Road', 'city': 'College Park', 'state': 'VA', 'zipCode': '10708', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Electronics', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 104, 'features': {'dimension': '10 inches', 'weight': '5 kg', 'extras': {'warranty': '1 year', 'color': 'Silver'}}}}}]}, {'category': 'Electronics', 'subCategory': 'Subcategory 2

In [53]:
# print pretty for question x

results_brief = eCommerce_collection.find({
    "viewedProducts.category": "Electronics",
    "cart.items.category": "Electronics"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "viewedProducts.category": "Electronics",
    "cart.items.category": "Electronics"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "viewedProducts.category": 1,
     "cart.items.category": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0024"
}
{
  "customerID": "CUST0024",
  "viewedProducts": [
    {
      "category": "Electronics"
    },
    {
      "category": "Electronics"
    },
    {
      "category": "Clothing"
    },
    {
      "category": "Footwear"
    }
  ],
  "cart": {
    "items": [
      {
        "category": "Electronics"
      },
      {
        "category": "Electronics"
      }
    ]
  }
}


In [55]:
## xi) Find customers who have viewed Clothing products, have Books in their wishlist and live in Hyattsville.

results = eCommerce_collection.find({
    "viewedProducts.category": "Clothing",
    "wishlist.category": "Books",
    "personalInfo.contact.address.city": "Hyattsville"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c110c'), 'customerID': 'CUST0013', 'personalInfo': {'name': {'firstName': 'Prady_13', 'lastName': 'Mohta_13'}, 'contact': {'email': 'prady13@example.com', 'phone': '507-555-4005', 'address': {'street': '366 Guliford Road', 'city': 'Hyattsville', 'state': 'MD', 'zipCode': '83986', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Clothing', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 338, 'features': {'dimension': '9 inches', 'weight': '4 kg', 'extras': {'warranty': '1 year', 'color': 'Black'}}}}}]}, {'category': 'Electronics', 'subCategory': 'Subcategory 2', 'it

In [56]:
# print pretty for question xi

results_brief = eCommerce_collection.find({
    "viewedProducts.category": "Clothing",
    "wishlist.category": "Books",
    "personalInfo.contact.address.city": "Hyattsville"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "viewedProducts.category": "Clothing",
    "wishlist.category": "Books",
    "personalInfo.contact.address.city": "Hyattsville"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "viewedProducts.category": 1,
     "wishlist.category": 1,
     "personalInfo.contact.address.city": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0013"
}
{
  "customerID": "CUST0021"
}
{
  "customerID": "CUST0013",
  "personalInfo": {
    "contact": {
      "address": {
        "city": "Hyattsville"
      }
    }
  },
  "viewedProducts": [
    {
      "category": "Clothing"
    },
    {
      "category": "Electronics"
    },
    {
      "category": "Footwear"
    }
  ],
  "wishlist": [
    {
      "category": "Electronics"
    },
    {
      "category": "Footwear"
    },
    {
      "category": "Electronics"
    },
    {
      "category": "Books"
    }
  ]
}
{
  "customerID": "CUST0021",
  "personalInfo": {
    "contact": {
      "address": {
        "city": "Hyattsville"
      }
    }
  },
  "viewedProducts": [
    {
      "category": "Clothing"
    },
    {
      "category": "Electronics"
    },
    {
      "category": "Home Appliances"
    }
  ],
  "wishlist": [
    {
      "category": "Books"
    },
    {
      "category": "Electronics"
    },
    {
      "category": "Clothing"
    },
    {
      "cate

In [59]:
## xii) Find customers who have placed an order in the year 2023, their cart items have a warranty and cart items have a category of Home Appliances.

results = eCommerce_collection.find({
    "orderHistory.orderDate": {"$regex": "^2023"},
    "orderHistory.products.items.details.specs.features.extras.warranty": {"$exists": "true"},
    "cart.items.category": "Home Appliances"
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c1100'), 'customerID': 'CUST0001', 'personalInfo': {'name': {'firstName': 'Prady_1', 'lastName': 'Mohta_1'}, 'contact': {'email': 'prady1@example.com', 'phone': '810-555-5903', 'address': {'street': '968 Guliford Road', 'city': 'Bethesda', 'state': 'VA', 'zipCode': '49836', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Clothing', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 372, 'features': {'dimension': '6 inches', 'weight': '3 kg', 'extras': {'warranty': '1 year', 'color': 'White'}}}}}]}, {'category': 'Footwear', 'subCategory': 'Subcategory 2', 'items': [{'

In [61]:
# print pretty for question xii

results_brief = eCommerce_collection.find({
    "orderHistory.orderDate": {"$regex": "^2023"},
    "orderHistory.products.items.details.specs.features.extras.warranty": {"$exists": "true"},
    "cart.items.category": "Home Appliances"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "orderHistory.orderDate": {"$regex": "^2023"},
    "orderHistory.products.items.details.specs.features.extras.warranty": {"$exists": "true"},
    "cart.items.category": "Home Appliances"},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "orderHistory.orderDate": 1,
     "orderHistory.products.items.details.specs.features.extras.warranty": 1,
     "cart.items.category": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0001"
}
{
  "customerID": "CUST0005"
}
{
  "customerID": "CUST0007"
}
{
  "customerID": "CUST0009"
}
{
  "customerID": "CUST0012"
}
{
  "customerID": "CUST0019"
}
{
  "customerID": "CUST0021"
}
{
  "customerID": "CUST0025"
}
{
  "customerID": "CUST0027"
}
{
  "customerID": "CUST0001",
  "cart": {
    "items": [
      {
        "category": "Home Appliances"
      },
      {
        "category": "Books"
      }
    ]
  },
  "orderHistory": [
    {
      "orderDate": "2023-09-01",
      "products": [
        {
          "items": [
            {
              "details": {
                "specs": {
                  "features": {
                    "extras": {
                      "warranty": "1 year"
                    }
                  }
                }
              }
            }
          ]
        },
        {
          "items": [
            {
              "details": {
                "specs": {
                  "features": {
                    "extr

In [63]:
## xiii) Find customers who live in VA, have viewed products of Electronics and have a product less than $200 in their cart.

results = eCommerce_collection.find({
    "personalInfo.contact.address.state": "VA",
    "viewedProducts.category": "Electronics",
    "cart.items.items.details.specs.price": {"$lt": 200}
})

for result in results:
    print(result)

{'_id': ObjectId('68f45a4f0ba48fd48e8c1106'), 'customerID': 'CUST0007', 'personalInfo': {'name': {'firstName': 'Prady_7', 'lastName': 'Mohta_7'}, 'contact': {'email': 'prady7@example.com', 'phone': '731-555-5317', 'address': {'street': '284 Guliford Road', 'city': 'Bethesda', 'state': 'VA', 'zipCode': '61151', 'country': 'USA', 'deliveryDetails': {'instructions': 'Leave at front door', 'timePreferences': {'weekdays': 'Afternoon', 'weekends': 'Morning', 'specialInstructions': {'ifNotHome': 'Leave with neighbor', 'neighborDetails': {'name': 'Manav Gupta', 'contactNumber': '333-444-5555', 'address': '4321 Hartwick Road'}}}}}}}, 'viewedProducts': [{'category': 'Home Appliances', 'subCategory': 'Subcategory 1', 'items': [{'name': 'Product 1', 'brand': 'Brand 1', 'details': {'model': 'Model 1', 'specs': {'price': 112, 'features': {'dimension': '13 inches', 'weight': '5 kg', 'extras': {'warranty': '1 year', 'color': 'White'}}}}}]}, {'category': 'Clothing', 'subCategory': 'Subcategory 2', 'ite

In [64]:
# print pretty for question xiii

results_brief = eCommerce_collection.find({
    "personalInfo.contact.address.state": "VA",
    "viewedProducts.category": "Electronics",
    "cart.items.items.details.specs.price": {"$lt": 200}},
    {"_id": 0,                          # Do not display _id
     "customerID": 1}                   # Display customerID only
)

results = eCommerce_collection.find({
    "personalInfo.contact.address.state": "VA",
    "viewedProducts.category": "Electronics",
    "cart.items.items.details.specs.price": {"$lt": 200}},
    {"_id": 0,                          # Do not display _id
     "customerID": 1,
     "personalInfo.contact.address.state": 1,
     "viewedProducts.category": 1,
     "cart.items.items.details.specs.price": 1}
)

for result in results_brief:
    print(json.dumps(result, indent=2, default=json_util.default))

for result in results:
    print(json.dumps(result, indent=2, default=json_util.default))

{
  "customerID": "CUST0007"
}
{
  "customerID": "CUST0007",
  "personalInfo": {
    "contact": {
      "address": {
        "state": "VA"
      }
    }
  },
  "viewedProducts": [
    {
      "category": "Home Appliances"
    },
    {
      "category": "Clothing"
    },
    {
      "category": "Footwear"
    },
    {
      "category": "Electronics"
    }
  ],
  "cart": {
    "items": [
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 551
              }
            }
          }
        ]
      },
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 653
              }
            }
          }
        ]
      },
      {
        "items": [
          {
            "details": {
              "specs": {
                "price": 161
              }
            }
          }
        ]
      }
    ]
  }
}
